<a href="https://colab.research.google.com/github/sudipto2104/ai-platform-engineering-portfolio/blob/main/02-ai-agents-platform-engineering/02-langgraph-ai-agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Clone your repo
!git clone https://github.com/sudipto2104/ai-platform-engineering-portfolio.git

# Go to Project 02 folder
%cd ai-platform-engineering-portfolio/02-ai-agents-platform-engineering

# Install dependencies
!pip install -q langchain langchain-openai langgraph langchain-community tavily-python python-dotenv gradio

Cloning into 'ai-platform-engineering-portfolio'...
remote: Enumerating objects: 105, done.
remote: Counting objects: 100% (105/105), done.
remote: Compressing objects: 100% (98/98), done.
remote: Total 105 (delta 34), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (105/105), 39.69 KiB | 5.67 MiB/s, done.
Resolving deltas: 100% (34/34), done.
/content/ai-platform-engineering-portfolio/02-ai-agents/ai-platform-engineering-portfolio/02-ai-agents-platform-engineering


Load API Keys

In [5]:
import os
from google.colab import userdata

# Load your existing OpenAI key
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

print("✅ OpenAI API Key loaded successfully!")

✅ OpenAI API Key loaded successfully!


In [8]:
import os
from google.colab import userdata
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langgraph.prebuilt import create_react_agent
from langchain_community.tools.tavily_search import TavilySearchResults

# Ensure TAVILY_API_KEY is loaded for this cell's execution context
os.environ["TAVILY_API_KEY"] = userdata.get('TAVILY_API_KEY')

# Initialize LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Tool
search_tool = TavilySearchResults(max_results=3)
tools = [search_tool]

# Create Agent
agent_executor = create_react_agent(llm, tools)

def ask_agent(question: str):
    print(f"\n❓ Question: {question}")
    print("-" * 60)

    response = agent_executor.invoke({"messages": [HumanMessage(content=question)]})
    answer = response["messages"][-1].content

    print(f"💡 Answer: {answer}\n")
    return answer

In [9]:
print("🤖 Testing AI Agent...\n")

ask_agent("What is Cilium?")
ask_agent("How is it different from standard Kubernetes Network Policies?")

🤖 Testing AI Agent...


❓ Question: What is Cilium?
------------------------------------------------------------
💡 Answer: Cilium is an open-source technology designed for networking, observability, and security in cloud-native environments, particularly those using Kubernetes. It leverages eBPF (Extended Berkeley Packet Filter) technology in the Linux kernel to provide advanced networking capabilities.

### Key Features of Cilium:
- **Networking**: Cilium provides a flat Layer 3 network for containers and supports advanced features like BGP (Border Gateway Protocol) and service mesh capabilities. It can replace kube-proxy and offers distributed load balancing for traffic between pods and external services.
- **Security**: It allows for fine-grained network and application-level security policies, enabling developers to define and enforce these policies effectively.
- **Observability**: Cilium includes components like Hubble for network observability and Tetragon for security observabi

"Kubernetes Network Policies differ from standard network policies in several key ways:\n\n1. **Scope and Purpose**:\n   - **Kubernetes Network Policies**: These are specific to Kubernetes and are designed to control traffic between pods within a Kubernetes cluster. They are implemented as API objects and are enforced by the cluster's Container Network Interface (CNI) plugin (e.g., Calico, Cilium). The primary goal is to secure pod-to-pod and pod-to-endpoint communication based on workload identity rather than static IP addresses.\n   - **Standard Network Policies**: These are broader organizational concepts that encompass various network security measures, including perimeter firewalls, VPN access, and routing rules. They are not limited to containerized environments and can apply to traditional network setups.\n\n2. **Dynamic Nature**:\n   - Kubernetes operates in a dynamic environment where pods are ephemeral, meaning their IP addresses can change frequently. Standard network polici

In [10]:
import gradio as gr

def chat(message, history):
    if not message:
        return "", history
    response = ask_agent(message)
    history.append((message, response))
    return "", history

with gr.Blocks(title="Platform Engineering AI Agent") as demo:
    gr.Markdown("# 🤖 Platform Engineering AI Agent\nPowered by LangGraph + Tavily")

    chatbot = gr.Chatbot(height=550)
    msg = gr.Textbox(placeholder="Ask anything about Kubernetes, Cilium, Platform Engineering...", label="Your Question")

    msg.submit(chat, inputs=[msg, chatbot], outputs=[msg, chatbot])

demo.launch(share=True)

/tmp/ipykernel_19581/964005117.py:13: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=550)
/tmp/ipykernel_19581/964005117.py:13: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(height=550)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://25ad23df122adf959e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [14]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.prebuilt import create_react_agent
from langchain_community.tools.tavily_search import TavilySearchResults
from dotenv import load_dotenv

load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

search_tool = TavilySearchResults(max_results=3)
tools = [search_tool]

system_message = SystemMessage(
    content="""You are a helpful Platform Engineering AI Assistant.
You have access to web search.
Always maintain the full conversation context.
Refer back to previous messages when relevant."""
)

agent_executor = create_react_agent(llm, tools, state_modifier=system_message)

# Global memory store for simplicity in Colab
memory = {}

def ask_agent(question: str, thread_id: str = "default"):
    print(f"\n❓ Question: {question}")
    print("-" * 70)

    # Get previous messages from memory
    if thread_id not in memory:
        memory[thread_id] = []

    # Add new question
    memory[thread_id].append(HumanMessage(content=question))

    response = agent_executor.invoke({
        "messages": memory[thread_id]
    })

    answer = response["messages"][-1].content

    # Save the answer to memory
    memory[thread_id].append(response["messages"][-1])

    print(f"💡 Answer: {answer}\n")
    return answer


if __name__ == "__main__":
    print("🤖 Agent with Memory Ready!\n")

    ask_agent("What is Cilium?", "demo")
    ask_agent("How does it compare to standard Kubernetes Network Policies?", "demo")
    ask_agent("Summarize our conversation about Cilium", "demo")

🤖 Agent with Memory Ready!


❓ Question: What is Cilium?
----------------------------------------------------------------------
💡 Answer: Cilium is an open-source networking and security project that provides a way to manage and secure network connectivity between containerized applications, particularly in Kubernetes environments. It is built on top of the Linux kernel's eBPF (extended Berkeley Packet Filter) technology, which allows for high-performance networking and security features without the need for traditional kernel modules.

Key features of Cilium include:

1. **Network Security**: Cilium enables fine-grained network policies that can control traffic between services based on various attributes, such as labels, namespaces, and more.

2. **Load Balancing**: It provides load balancing capabilities for services, ensuring that traffic is distributed evenly across multiple instances of an application.

3. **Visibility and Monitoring**: Cilium offers tools for monitoring and obse

In [17]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.prebuilt import create_react_agent
from langchain_community.tools.tavily_search import TavilySearchResults
from dotenv import load_dotenv

load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

search_tool = TavilySearchResults(max_results=3)
tools = [search_tool]

system_message = SystemMessage(
    content="""You are a helpful Platform Engineering AI Assistant.
You have access to web search tools.
Think step by step. Use tools when needed.
Keep conversation context."""
)

agent_executor = create_react_agent(llm, tools, state_modifier=system_message)

def ask_agent(question: str, thread_id: str = "default"):
    print(f"\n❓ Question: {question}")
    print("-" * 70)

    response = agent_executor.invoke({
        "messages": [HumanMessage(content=question)],
        "config": {"configurable": {"thread_id": thread_id}}
    })

    # Safe way to extract the answer
    final_message = response["messages"][-1]
    answer = final_message.content if hasattr(final_message, 'content') else str(final_message)

    print(f"💡 Answer: {answer}\n")
    return answer


if __name__ == "__main__":
    print("🤖 Agent with Memory Ready!\n")
    ask_agent("What is Cilium?", "test")

🤖 Agent with Memory Ready!


❓ Question: What is Cilium?
----------------------------------------------------------------------
💡 Answer: Cilium is an open-source networking and security project that provides a way to manage and secure network connectivity between containerized applications, particularly in Kubernetes environments. It is built on top of the Linux kernel's eBPF (extended Berkeley Packet Filter) technology, which allows for high-performance networking and security features without the need for traditional kernel modules.

Key features of Cilium include:

1. **Network Security**: Cilium enables fine-grained network security policies that can be applied to individual workloads, allowing for better control over traffic flow and enhanced security.

2. **Load Balancing**: It provides advanced load balancing capabilities, allowing for efficient distribution of network traffic across multiple instances of services.

3. **Visibility and Monitoring**: Cilium offers tools for moni

In [18]:
import gradio as gr
from agent import ask_agent
import os
from dotenv import load_dotenv

load_dotenv()

print("🚀 Starting Platform Engineering AI Agent with Memory...")

# Use a fixed thread_id for the entire chat session
THREAD_ID = "gradio-session-1"

def chat(message, history):
    if not message or message.strip() == "":
        return "", history

    try:
        # Pass the thread_id to maintain memory across the chat
        response = ask_agent(message, thread_id=THREAD_ID)
        history.append((message, response))
        return "", history
    except Exception as e:
        error_msg = f"❌ Sorry, something went wrong: {str(e)}"
        history.append((message, error_msg))
        return "", history


with gr.Blocks(title="Platform Engineering AI Agent", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🤖 Platform Engineering AI Agent\n**LangGraph + Memory + Web Search**")

    gr.Markdown("Ask anything about Kubernetes, Cilium, ArgoCD, Backstage, Platform Engineering, etc.")

    chatbot = gr.Chatbot(
        height=650,
        show_copy_button=True,
        avatar_images=["🧑‍💻", "🤖"]
    )

    msg = gr.Textbox(
        placeholder="Type your question here...",
        label="Your Message",
        lines=2
    )

    with gr.Row():
        submit_btn = gr.Button("Send", variant="primary", scale=4)
        clear_btn = gr.Button("Clear Chat", variant="secondary")

    submit_btn.click(
        chat,
        inputs=[msg, chatbot],
        outputs=[msg, chatbot]
    )

    clear_btn.click(
        lambda: ([], []),  # Clear history
        None,
        [chatbot],
        queue=False
    )

demo.launch(share=True)

🚀 Starting Platform Engineering AI Agent with Memory...


/tmp/ipykernel_19581/1131662956.py:28: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Platform Engineering AI Agent", theme=gr.themes.Soft()) as demo:
/tmp/ipykernel_19581/1131662956.py:33: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/tmp/ipykernel_19581/1131662956.py:33: DeprecationWarning: The 'show_copy_button' parameter will be removed in Gradio 6.0. You will need to use 'buttons=["copy"]' instead.
  chatbot = gr.Chatbot(
/tmp/ipykernel_19581/1131662956.py:33: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b9697313d50b638f61.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
